# Rally Gemma4 E4B Compare

Scores **Google base**, **Heretic direct**, and **RP A100/B75** on the same 4-prompt gate plus 100-prompt adult false-refusal probe.


In [ ]:
import os, platform, shutil
try:
    import torch
    print('torch=', torch.__version__, 'gpus=', torch.cuda.device_count())
except Exception as exc:
    print('torch_probe_error=', repr(exc))
print('working_disk_free_gb=', round(shutil.disk_usage('/kaggle/working').free / 1024**3, 2))

In [ ]:
import os, subprocess, sys, time
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
for attempt in range(5):
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('HF_TOKEN')
        os.environ.setdefault('HF_TOKEN', token)
        os.environ.setdefault('HUGGING_FACE_HUB_TOKEN', token)
        break
    except Exception:
        time.sleep(3)
packages = [
    'git+https://github.com/huggingface/transformers.git',
    'accelerate>=1.13.0', 'peft>=0.19.0', 'safetensors>=0.7.0',
    'huggingface_hub[hf_transfer]>=1.5.0', 'bitsandbytes>=0.49.0',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', *packages])

In [ ]:
import os, subprocess, sys
from pathlib import Path
REPO_URL = os.environ.get('HERETIC_TO_ONNX_REPO', 'https://github.com/alkahest-ai/heretic-to-onnx.git')
REPO_REF = os.environ.get('HERETIC_TO_ONNX_REF', 'codex/kaggle-heretic-2b-run')
REPO_DIR = Path('/kaggle/working/heretic-to-onnx')
if REPO_DIR.exists():
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_REF])
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'checkout', REPO_REF])
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'])
else:
    subprocess.check_call(['git', 'clone', '--branch', REPO_REF, '--depth', '1', REPO_URL, str(REPO_DIR)])
print('repo_head=', subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip())
import shutil, sys
subprocess.check_call([sys.executable, str(REPO_DIR / 'scripts/kaggle_disk_cleanup.py'), '--root', '/kaggle/working'])
shutil.rmtree(REPO_DIR / '.git', ignore_errors=True)
print('working_disk_free_gb_after_cleanup=', round(shutil.disk_usage('/kaggle/working').free / 1024**3, 2))

In [ ]:
import os, subprocess, sys
from pathlib import Path
os.environ.setdefault('RALLY_SCORECARD_LOAD_IN_4BIT', '1')
REPO_DIR = Path('/kaggle/working/heretic-to-onnx')
sys.path.insert(0, str(REPO_DIR))
from scripts.kaggle_rally_artifacts import find_artifacts

artifact_name = os.environ.get('RALLY_TWO_STAGE_ARTIFACT_NAME', 'rally-e4b-two-stage-sft')
direct_id = os.environ.get('RALLY_HERETIC_MODEL_ID', 'coder3101/gemma-4-E4B-it-heretic')
base_id = os.environ.get('RALLY_BASE_MODEL_ID', 'google/gemma-4-E4B-it')
work_dir = Path('/kaggle/working/rally-e4b-heretic-compare')
work_dir.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, str(REPO_DIR / 'scripts/kaggle_rally_model_compare_scorecard.py'),
    '--work-dir', str(work_dir),
    '--report-path', str(work_dir / 'rally-e4b-heretic-compare-report.json'),
    '--models', f'base:{base_id},heretic:{direct_id}',
    '--refusal-probe-count', os.environ.get('RALLY_REFUSAL_PROBE_COUNT', '100'),
    '--load-in-4bit',
]
try:
    artifacts = find_artifacts(os.environ.get('RALLY_ARTIFACT_DIR', ''), artifact_name)
    cmd.extend([
        '--rp-name', 'rp-a100-b75',
        '--rp-base-model-id', direct_id,
        '--rp-stage-a-adapter', str(artifacts / 'stage-a-adapter'),
        '--rp-stage-b-adapter', str(artifacts / 'stage-b-adapter'),
        '--rp-stage-b-scale', os.environ.get('RALLY_STAGE_B_SCALE', '0.75'),
    ])
    print('rp_adapter_inference=', artifacts)
except Exception as exc:
    print('rp_adapter_skipped=', type(exc).__name__, exc)
subprocess.check_call(cmd)


In [ ]:
import json
from pathlib import Path
report_path = Path('/kaggle/working/rally-e4b-heretic-compare/rally-e4b-heretic-compare-report.json')
if report_path.exists():
    report = json.loads(report_path.read_text())
    print(json.dumps({'ranking': report.get('ranking'), 'models': list((report.get('models') or {}).keys())}, indent=2))
